# Models and decision rules

Decision tree, random forest and XGBoost are tuned on a validation period with `RandomizedSearchCV` and a predefined time split. Thresholds are chosen on validation data only.

In [1]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from IPython.display import Image, Markdown, display

from src import config, features, models, pipeline, simulation

In [2]:
FAST = os.environ.get("NOTEBOOK_FAST", "0") == "1"
df = features.load_modeling_frame()
splits = models.time_splits(df)
{name: int(mask.sum()) for name, mask in splits.items()}

{'train': 445871, 'valid': 165530, 'test': 471437, 'usable': 1091251}

In [3]:
result = models.train_all(df, fast=FAST)
result["best_params"]

{'dt': {'min_samples_leaf': 2000, 'max_depth': 8, 'criterion': 'gini'},
 'rf': {'n_estimators': 80,
  'min_samples_leaf': 100,
  'max_features': 'sqrt',
  'max_depth': 8},
 'xgb': {'subsample': 0.8,
  'n_estimators': 60,
  'min_child_weight': 50,
  'max_depth': 3,
  'learning_rate': 0.05,
  'colsample_bytree': 0.7}}

In [4]:
result["metrics"]

,split,prediction,rows,base_rate,auc,average_precision,precision_top_1pct
0,valid,pred_dt,165530,0.260811,0.518134,0.273614,0.303323
1,valid,pred_rf,165530,0.260811,0.555086,0.298272,0.367372
2,valid,pred_xgb,165530,0.260811,0.547537,0.292994,0.370393
3,valid,pred_ens,165530,0.260811,0.554385,0.297867,0.366767
4,test,pred_dt,471437,0.274298,0.529577,0.294496,0.308019
5,test,pred_rf,471437,0.274298,0.553771,0.313263,0.390751
6,test,pred_xgb,471437,0.274298,0.550387,0.311643,0.404752
7,test,pred_ens,471437,0.274298,0.554355,0.314096,0.403055


## Decision rule thresholds (chosen on validation)

In [5]:
pd.DataFrame(result["thresholds"]).T

,threshold,quantile,valid_rows_selected,valid_mean_net_return,tstat
pred_dt,0.326411,0.9,18366.0,0.002784,5.205735
pred_rf,0.301748,0.9,16553.0,0.006423,10.652110
pred_xgb,0.302220,0.9,16553.0,0.003394,5.890694
pred_ens,0.300476,0.9,16553.0,0.005110,8.462615


## Feature importance

In [6]:
result["importance"].groupby("model").head(15)

,feature,importance,model
0,vol_63,0.033973,rf
1,vix_pctile_252,0.028359,rf
2,ret_63,0.028297,rf
3,atr_pct_14,0.027903,rf
4,cs_rank_vol_21,0.025570,rf
5,ew_dist_sma_200,0.025041,rf
6,ret_126,0.024060,rf
7,vol_21,0.022032,rf
8,vix,0.021850,rf
9,ew_vol_21,0.021718,rf
